# Notebook 1 — Obtenção de Dados Olist
*Fonte utilizada: Olist (Kaggle)	Dados reais de e-commerce brasileiro (Vendas, Clientes, Produtos, Filiais)	Kaggle - Olist*


Nota: Os dados de transações, produtos e localização são reais. Atributos demográficos (Sexo e Idade) foram enriquecidos via script para cumprir os requisitos de modelagem solicitados no Business Case

In [ ]:
#!pip install requests pandas ydata-profiling --quiet


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [20]:
import pandas as pd
import sqlite3
import os
import kagglehub
import numpy as np

c:\Users\joaol\Downloads\gocase-v2\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1.1 — Download dos dados via Kagglehub

In [24]:
BASE_DIR = ".."
DATA_DIR = os.path.join(BASE_DIR, "dados")
DB_PATH = os.path.join(DATA_DIR, "varejo_gocase.db")
os.makedirs(DATA_DIR, exist_ok=True)

def download_data():
    print(" Baixando dataset da Olist via KaggleHub...")
    path = kagglehub.dataset_download("olistbr/brazilian-ecommerce")
    print(f" Arquivos baixados em: {path}")
    return path


path=download_data()
path

 Baixando dataset da Olist via KaggleHub...
 Arquivos baixados em: C:\Users\joaol\.cache\kagglehub\datasets\olistbr\brazilian-ecommerce\versions\2


'C:\\Users\\joaol\\.cache\\kagglehub\\datasets\\olistbr\\brazilian-ecommerce\\versions\\2'

# 1.2 — ETL dos dados

In [23]:
def transform_and_load(path): 

    """ ETL: Extração, Transformação e Carga dos dados da Olist"""
    
    customers = pd.read_csv(f"{path}/olist_customers_dataset.csv")
    items = pd.read_csv(f"{path}/olist_order_items_dataset.csv")
    orders = pd.read_csv(f"{path}/olist_orders_dataset.csv")
    products = pd.read_csv(f"{path}/olist_products_dataset.csv")
    sellers = pd.read_csv(f"{path}/olist_sellers_dataset.csv")
    product_category = pd.read_csv(f"{path}/product_category_name_translation.csv")

    print("Iniciando Normalização...")

    #  TABELA FILIAIS 
    # O case pede: ID da filial, Nome, Localização (cidade/uf)
    df_filiais = sellers[['seller_id', 'seller_city', 'seller_state']].copy()
    df_filiais.columns = ['ID_Filial', 'Cidade', 'Estado']
    df_filiais['Nome'] = "Filial " + df_filiais['Estado'] + " - " + df_filiais['ID_Filial'].str[:4]
    df_filiais['Localizacao'] = df_filiais['Cidade'] + "/" + df_filiais['Estado']
    
    #  
    # O case pede: ID, Nome, Categoria, Preço
    # Traduzindo categorias para inglês/português legível
    products = products.merge(product_category, on='product_category_name', how='left')
    df_produtos = products[['product_id', 'product_category_name_english']].copy()
    df_produtos.columns = ['ID_Produto', 'Categoria']
    df_produtos['Nome'] = "Produto " + df_produtos['ID_Produto'].str[:5]
    df_produtos['Categoria'] = df_produtos['Categoria'].fillna('Outros')
    
    # Preço médio por produto (extraído da tabela de itens)
    precos = items.groupby('product_id')['price'].mean().reset_index()
    df_produtos = df_produtos.merge(precos, left_on='ID_Produto', right_on='product_id').drop(columns='product_id')
    df_produtos.rename(columns={'price': 'Preco'}, inplace=True)

    #  TABELA CLIENTES (Enriquecendo com dados FICTÍCIOS(!!!!) para o Case) 
    # O case pede: ID, Nome, Sexo, Idade, Região
    df_clientes = customers[['customer_id', 'customer_state']].copy()
    df_clientes.columns = ['ID_Cliente', 'Estado']
    
    # Gerando dados sintéticos para cumprir o requisito do Business Case
    np.random.seed(42)
    df_clientes['Nome'] = "Cliente " + df_clientes['ID_Cliente'].str[:5]
    df_clientes['Sexo'] = np.random.choice(['M', 'F'], size=len(df_clientes))
    df_clientes['Idade'] = np.random.randint(18, 75, size=len(df_clientes))
    
    # Mapeando Estados para Regiões
    regioes = {
        'SP': 'Sudeste', 'RJ': 'Sudeste', 'MG': 'Sudeste', 'ES': 'Sudeste',
        'PR': 'Sul', 'SC': 'Sul', 'RS': 'Sul',
        'MT': 'Centro-Oeste', 'MS': 'Centro-Oeste', 'GO': 'Centro-Oeste', 'DF': 'Centro-Oeste',
        'BA': 'Nordeste', 'PE': 'Nordeste', 'CE': 'Nordeste', 'RN': 'Nordeste', 'PB': 'Nordeste', 'AL': 'Nordeste', 'SE': 'Nordeste', 'MA': 'Nordeste', 'PI': 'Nordeste',
        'AM': 'Norte', 'PA': 'Norte', 'RO': 'Norte', 'RR': 'Norte', 'AC': 'Norte', 'TO': 'Norte', 'AP': 'Norte'
    }
    df_clientes['Regiao'] = df_clientes['Estado'].map(regioes)

    # VENDAS
    # O case pede: ID_Venda, ID_Cliente, ID_Produto, Data, Quantidade, Valor_Total
    # Join entre Orders e Items
    df_vendas = items.merge(orders, on='order_id')
    df_vendas = df_vendas.merge(sellers[['seller_id']], on='seller_id') # Garantindo que temos a filial
    
    df_vendas = df_vendas[['order_id', 'customer_id', 'product_id', 'seller_id', 'order_purchase_timestamp', 'price']]
    df_vendas.columns = ['ID_Venda', 'ID_Cliente', 'ID_Produto', 'ID_Filial', 'Data_Venda', 'Valor_Total']
    df_vendas['Data_Venda'] = pd.to_datetime(df_vendas['Data_Venda'])
    df_vendas['Quantidade'] = 1 # Olist registra cada item em uma linha

    
    print(f" Salvando dados em {DB_PATH}...")
    conn = sqlite3.connect(DB_PATH)
    
    df_filiais.to_sql('Filiais', conn, if_exists='replace', index=False)
    df_produtos.to_sql('Produtos', conn, if_exists='replace', index=False)
    df_clientes.to_sql('Clientes', conn, if_exists='replace', index=False)
    df_vendas.to_sql('Vendas', conn, if_exists='replace', index=False)
    
    conn.close()
    
    # Salvando CSVs para o app (conforme pedido no case)
    df_filiais.to_csv(f"{DATA_DIR}/filiais.csv", index=False)
    df_produtos.to_csv(f"{DATA_DIR}/produtos.csv", index=False)
    df_clientes.to_csv(f"{DATA_DIR}/clientes.csv", index=False)
    df_vendas.to_csv(f"{DATA_DIR}/vendas.csv", index=False)
    
    print("ETL:OK")

In [25]:
transform_and_load(path)

Iniciando Normalização...
 Salvando dados em ..\dados\varejo_gocase.db...
ETL:OK


# Verificação da qualidades dos dados

In [31]:
path_clientes = os.path.join(DATA_DIR, "clientes.csv")
path_filiais = os.path.join(DATA_DIR, "filiais.csv")
path_produtos = os.path.join(DATA_DIR, "produtos.csv")
path_vendas = os.path.join(DATA_DIR, "vendas.csv")

data = {
    "clientes": pd.read_csv(path_clientes),
    "filiais": pd.read_csv(path_filiais),
    "produtos": pd.read_csv(path_produtos),
    "vendas": pd.read_csv(path_vendas)
}

for nome, df in data.items():
    print(f" Analisando: {nome} - Linhas: {df.shape[0]}, Colunas: {df.shape[1]}")
    print(f"Colunas: {df.columns.tolist()}")
    print(f"Nulos:\n{df.isnull().sum()[df.isnull().sum() > 0]}")
    print(f"Duplicatas: {df.duplicated().sum()}")

    display(df.head(5))
    
    print("-" * 30)

 Analisando: clientes - Linhas: 99441, Colunas: 6
Colunas: ['ID_Cliente', 'Estado', 'Nome', 'Sexo', 'Idade', 'Regiao']
Nulos:
Series([], dtype: int64)
Duplicatas: 0


,ID_Cliente,Estado,Nome,Sexo,Idade,Regiao
0,06b8999e2fba1a1fbc88172c00ba8bc7,SP,Cliente 06b89,M,59,Sudeste
1,18955e83d337fd6b2def6b18a428ac77,SP,Cliente 18955,F,53,Sudeste
2,4e7b3e00288586ebd08712fdd0374a03,SP,Cliente 4e7b3,M,42,Sudeste
3,b2b6027bc5c5109e529d4dc6358b12c3,SP,Cliente b2b60,M,69,Sudeste
4,4f2d8ab171c80ec8364f7c12e35b23ad,SP,Cliente 4f2d8,M,54,Sudeste


------------------------------
 Analisando: filiais - Linhas: 3095, Colunas: 5
Colunas: ['ID_Filial', 'Cidade', 'Estado', 'Nome', 'Localizacao']
Nulos:
Series([], dtype: int64)
Duplicatas: 0


,ID_Filial,Cidade,Estado,Nome,Localizacao
0,3442f8959a84dea7ee197c632cb2df15,campinas,SP,Filial SP - 3442,campinas/SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,mogi guacu,SP,Filial SP - d1b6,mogi guacu/SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,rio de janeiro,RJ,Filial RJ - ce3a,rio de janeiro/RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,sao paulo,SP,Filial SP - c0f3,sao paulo/SP
4,51a04a8a6bdcb23deccc82b0b80742cf,braganca paulista,SP,Filial SP - 51a0,braganca paulista/SP


------------------------------
 Analisando: produtos - Linhas: 32951, Colunas: 4
Colunas: ['ID_Produto', 'Categoria', 'Nome', 'Preco']
Nulos:
Series([], dtype: int64)
Duplicatas: 0


,ID_Produto,Categoria,Nome,Preco
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumery,Produto 1e9e8,10.91
1,3aa071139cb16b67ca9e5dea641aaa2f,art,Produto 3aa07,248.00
2,96bd76ec8810374ed1b65e291975717f,sports_leisure,Produto 96bd7,79.80
3,cef67bcfe19066a932b7673e239eb23d,baby,Produto cef67,112.30
4,9dc1a7de274444849c219cff195d0b71,housewares,Produto 9dc1a,37.90


------------------------------
 Analisando: vendas - Linhas: 112650, Colunas: 7
Colunas: ['ID_Venda', 'ID_Cliente', 'ID_Produto', 'ID_Filial', 'Data_Venda', 'Valor_Total', 'Quantidade']
Nulos:
Series([], dtype: int64)
Duplicatas: 10225


,ID_Venda,ID_Cliente,ID_Produto,ID_Filial,Data_Venda,Valor_Total,Quantidade
0,00010242fe8c5a6d1ba2dd792cb16214,3ce436f183e68e07877b285a838db11a,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-13 08:59:02,58.90,1
1,00018f77f2f0320c557190d7a144bdd3,f6dd3ec061db4e3987629fe6b26e5cce,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-04-26 10:53:06,239.90,1
2,000229ec398224ef6ca0657da4fc703e,6489ae5e4333f3693df5ad4372dab6d3,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-14 14:33:31,199.00,1
3,00024acbcdf0a6daa1e931b038114c75,d4eb9395c8c0431ee92fce09860c5a06,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-08 10:00:35,12.99,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,58dbd0b2d70206bf40e62cd34e84d795,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-04 13:57:51,199.90,1


------------------------------


---
##  Checkpoint


**Próximo:** `02_analise_sql.ipynb`